In [ ]:
import cv2
import os
import time
import json
import numpy as np
import mediapipe as mp
import pandas as pd

# ---------------- Config ----------------
CONFIG_FILE = "config.json"
if not os.path.exists(CONFIG_FILE):
    default_config = {
        "SAVE_PATH": "dataset",
        "FPS": 30,
        "VIDEO_DURATION": 7,
        "PAUSE_TIME": 2,
        "VIDEO_PAUSE": 5,
        "STATIC_IMAGES_PER_GESTURE": 60,
        "DYNAMIC_VIDEOS_PER_GESTURE": 10
    }
    with open(CONFIG_FILE, "w", encoding="utf-8") as f:
        json.dump(default_config, f, indent=4, ensure_ascii=False)

with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    config = json.load(f)

SAVE_PATH = config["SAVE_PATH"]
FPS = config["FPS"]
VIDEO_DURATION = config["VIDEO_DURATION"]
PAUSE_TIME = config["PAUSE_TIME"]
VIDEO_PAUSE = config["VIDEO_PAUSE"]
STATIC_IMAGES_PER_GESTURE = config["STATIC_IMAGES_PER_GESTURE"]
DYNAMIC_VIDEOS_PER_GESTURE = config["DYNAMIC_VIDEOS_PER_GESTURE"]

# ---------------- MediaPipe ----------------
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False,
                       max_num_hands=2,
                       min_detection_confidence=0.7,
                       min_tracking_confidence=0.7)
mp_draw = mp.solutions.drawing_utils

# ---------------- Dataset & Metadata ----------------
os.makedirs(SAVE_PATH, exist_ok=True)
csv_path = os.path.join(SAVE_PATH, "dataset.csv")
json_path = os.path.join(SAVE_PATH, "dataset.json")
labels_path = os.path.join(SAVE_PATH, "labels.json")
npy_path = os.path.join(SAVE_PATH, "dataset.npy")

df = pd.read_csv(csv_path, encoding="utf-8-sig") if os.path.exists(csv_path) else pd.DataFrame(columns=["gesture","type","hand","file"])
metadata = json.load(open(json_path, "r", encoding="utf-8")) if os.path.exists(json_path) else {}
labels = json.load(open(labels_path, "r", encoding="utf-8")) if os.path.exists(labels_path) else {}

# ---------------- Helpers ----------------
def detect_hand_side(results):
    """Detect hand side with mirror correction"""
    if not results.multi_handedness:
        return "unknown"
    
    hands_detected = []
    for hand in results.multi_handedness:
        # MediaPipe returns "Left" for right hand in selfie/mirror view
        detected_label = hand.classification[0].label
        # Invert the label for correct display
        actual_hand = "right" if detected_label == "Left" else "left"
        hands_detected.append(actual_hand)
    
    return "both" if len(hands_detected) == 2 else hands_detected[0]

def save_metadata(gesture, gtype, hand, filepath):
    global df, metadata, labels
    df = pd.concat([df, pd.DataFrame([[gesture, gtype, hand, filepath]], columns=df.columns)], ignore_index=True)
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    if gesture not in metadata:
        metadata[gesture] = {"type": gtype, "files":[]}
    metadata[gesture]["files"].append({"hand":hand, "file":filepath})
    with open(json_path,"w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4, ensure_ascii=False)
    if gesture not in labels:
        labels[gesture] = len(labels)
    with open(labels_path,"w", encoding="utf-8") as f:
        json.dump(labels, f, indent=4, ensure_ascii=False)
    np.save(npy_path, df.to_numpy(), allow_pickle=True)

def undo_last_capture(last_file):
    if not last_file or not os.path.exists(last_file):
        print("Nothing to undo.")
        return None
    os.remove(last_file)
    print(f"🗑 Deleted {last_file}")
    global df, metadata
    df = df[df["file"] != last_file]
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    for g in metadata:
        metadata[g]["files"] = [f for f in metadata[g]["files"] if f["file"] != last_file]
    with open(json_path,"w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4, ensure_ascii=False)
    return None

def get_next_file_number(folder, prefix, extension):
    if not os.path.exists(folder):
        return 0
    existing = [int(f.split("_")[-1].split(".")[0]) for f in os.listdir(folder) 
                if f.startswith(prefix) and f.split("_")[-1].split(".")[0].isdigit()]
    return max(existing, default=-1) + 1

def is_hand_fully_in_frame(results, frame_width, frame_height):
    if not results.multi_hand_landmarks:
        return False
    margin = 10  # Add margin to avoid edge cases
    for hand in results.multi_hand_landmarks:
        for lm in hand.landmark:
            x_px = int(lm.x * frame_width)
            y_px = int(lm.y * frame_height)
            if x_px < margin or x_px >= frame_width-margin or y_px < margin or y_px >= frame_height-margin:
                return False
    return True

# ---------------- Static Capture ----------------
def capture_static(gesture, image_count=STATIC_IMAGES_PER_GESTURE):
    gesture_dir = os.path.join(SAVE_PATH, "static", gesture)
    os.makedirs(gesture_dir, exist_ok=True)
    cap = cv2.VideoCapture(0)
    if not cap.isOpened(): 
        print("Cannot access camera")
        return
    
    count = 0
    last_file = None
    last_capture_time = 0
    capture_delay = 0.3  # Delay between captures (seconds)
    paused = False
    
    print(f"\n Static capture for '{gesture}' ({image_count} images)")
    print("Controls: SPACE=pause, U=undo, Q=quit")

    while count < image_count:
        ret, frame = cap.read()
        if not ret: break
        
        # Flip frame for mirror view
        frame = cv2.flip(frame, 1)
        
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)
        display_frame = frame.copy()
        
        # Get hand side with mirror correction
        hand_side = detect_hand_side(results) if results.multi_hand_landmarks else "unknown"
        
        # Draw landmarks
        if results.multi_hand_landmarks:
            for lm in results.multi_hand_landmarks:
                mp_draw.draw_landmarks(display_frame, lm, mp_hands.HAND_CONNECTIONS)
        
        # Status display
        status_color = (0, 255, 0) if not paused else (0, 0, 255)
        status_text = "CAPTURING" if not paused else "PAUSED"
        cv2.putText(display_frame, f"{gesture} ({hand_side}) [{status_text}]", 
                   (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, status_color, 2)
        cv2.putText(display_frame, f"Count: {count}/{image_count}", 
                   (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        
        # Check if hand is in frame
        current_time = time.time()
        if (results.multi_hand_landmarks and 
            is_hand_fully_in_frame(results, frame.shape[1], frame.shape[0]) and 
            not paused and
            current_time - last_capture_time >= capture_delay):
            
            fname = f"img_{get_next_file_number(gesture_dir,'img','.png'):03d}.png"
            fpath = os.path.join(gesture_dir, fname)
            cv2.imwrite(fpath, frame)
            save_metadata(gesture, "static", hand_side, fpath)
            last_file = fpath
            count += 1
            last_capture_time = current_time
            print(f" Captured {count}/{image_count}")
            
        elif not results.multi_hand_landmarks:
            cv2.putText(display_frame, "No hand detected", (10, 90), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        elif not is_hand_fully_in_frame(results, frame.shape[1], frame.shape[0]):
            cv2.putText(display_frame, "Hand not fully in frame", (10, 90), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        cv2.imshow("Static Capture", display_frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"): 
            break
        elif key == ord("u") and last_file: 
            last_file = undo_last_capture(last_file)
            count = max(count-1, 0)
        elif key == ord(" "):  # Space bar to pause
            paused = not paused
            print("⏸ Paused" if paused else " Resumed")
            
    cap.release()
    cv2.destroyAllWindows()
    print(f" Static capture complete! Total: {count} images")

# ---------------- Dynamic Capture ----------------
def capture_dynamic(gesture, video_duration=VIDEO_DURATION):
    gesture_dir = os.path.join(SAVE_PATH, "dynamic", gesture)
    os.makedirs(gesture_dir, exist_ok=True)
    cap = cv2.VideoCapture(0)
    if not cap.isOpened(): 
        print(" Cannot access camera")
        return
        
    width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    last_file = None
    recorded = 0
    print(f"\n Dynamic capture for '{gesture}' ({DYNAMIC_VIDEOS_PER_GESTURE} videos of {video_duration}s each)")

    while recorded < DYNAMIC_VIDEOS_PER_GESTURE:
        fname = f"vid_{get_next_file_number(gesture_dir,'vid','.mp4'):03d}.mp4"
        fpath = os.path.join(gesture_dir, fname)
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        out = cv2.VideoWriter(fpath, fourcc, FPS, (width, height))
        
        print(f"\n Video {recorded+1}/{DYNAMIC_VIDEOS_PER_GESTURE}")
        print(f" Get ready... recording starts in {PAUSE_TIME}s")
        
        # Countdown
        for i in range(PAUSE_TIME, 0, -1):
            ret, frame = cap.read()
            if ret:
                frame = cv2.flip(frame, 1)
                cv2.putText(frame, f"Starting in: {i}", (width//2-100, height//2), 
                           cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 3)
                cv2.imshow("Dynamic Capture", frame)
                cv2.waitKey(1000)

        hand_detected = False
        start_time = None
        frames_written = 0

        while True:
            ret, frame = cap.read()
            if not ret: 
                break
                
            # Flip frame for mirror view
            frame = cv2.flip(frame, 1)
            
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands.process(rgb)
            display_frame = frame.copy()
            
            # Get hand side
            hand_side = detect_hand_side(results) if results.multi_hand_landmarks else "unknown"

            if results.multi_hand_landmarks and is_hand_fully_in_frame(results, frame.shape[1], frame.shape[0]):
                # Draw landmarks
                for lm in results.multi_hand_landmarks:
                    mp_draw.draw_landmarks(display_frame, lm, mp_hands.HAND_CONNECTIONS)
                    
                if not hand_detected:
                    hand_detected = True
                    start_time = time.time()
                    print(" Recording started!")
                    
                # Write frame
                out.write(frame)
                frames_written += 1
                
                # Progress display
                elapsed = time.time() - start_time
                progress = elapsed / video_duration
                bar_width = int(width * 0.8 * progress)
                cv2.rectangle(display_frame, (int(width*0.1), height-30), 
                             (int(width*0.1) + bar_width, height-10), (0, 255, 0), -1)
                cv2.rectangle(display_frame, (int(width*0.1), height-30), 
                             (int(width*0.9), height-10), (255, 255, 255), 2)
                
                cv2.putText(display_frame, f"{gesture} ({hand_side}) Recording: {elapsed:.1f}/{video_duration}s", 
                           (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                
                if elapsed >= video_duration:
                    break
                    
            else:
                if hand_detected:
                    # Hand left frame during recording
                    print(" Hand left frame, discarding video")
                    out.release()
                    if os.path.exists(fpath):
                        os.remove(fpath)
                    hand_detected = False
                    break
                else:
                    # Waiting for hand
                    cv2.putText(display_frame, "Position hand in frame", (10, 30), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
                    if not results.multi_hand_landmarks:
                        cv2.putText(display_frame, "No hand detected", (10, 60), 
                                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                    else:
                                                cv2.putText(display_frame, "Hand not fully in frame", (10, 60), 
                                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            cv2.imshow("Dynamic Capture", display_frame)
            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"): 
                out.release()
                if os.path.exists(fpath) and frames_written < FPS:  # Less than 1 second
                    os.remove(fpath)
                cap.release()
                cv2.destroyAllWindows()
                return

        out.release()
        
        # Save if recording was successful
        if hand_detected and frames_written >= FPS * 2:  # At least 2 seconds
            save_metadata(gesture, "dynamic", hand_side, fpath)
            last_file = fpath
            recorded += 1
            print(f" Video {recorded}/{DYNAMIC_VIDEOS_PER_GESTURE} saved!")
        else:
            if os.path.exists(fpath):
                os.remove(fpath)
            print(" Video discarded (too short or incomplete)")
            
        # Rest between videos
        if recorded < DYNAMIC_VIDEOS_PER_GESTURE:
            print(f"\n⏸ Resting {VIDEO_PAUSE}s before next video...")
            for i in range(VIDEO_PAUSE, 0, -1):
                ret, frame = cap.read()
                if ret:
                    frame = cv2.flip(frame, 1)
                    cv2.putText(frame, f"Next video in: {i}s", (width//2-100, height//2), 
                               cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                    cv2.putText(frame, f"Videos completed: {recorded}/{DYNAMIC_VIDEOS_PER_GESTURE}", 
                               (width//2-150, height//2+50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
                    cv2.imshow("Dynamic Capture", frame)
                    if cv2.waitKey(1000) & 0xFF == ord('q'):
                        cap.release()
                        cv2.destroyAllWindows()
                        return

    cap.release()
    cv2.destroyAllWindows()
    print(f" Dynamic capture complete! Total: {recorded} videos")

# ---------------- Statistics Function ----------------
def show_statistics():
    """Display dataset statistics"""
    print("\n Dataset Statistics:")
    print("="*50)
    
    if df.empty:
        print("No data collected yet.")
        return
    
    # Overall stats
    print(f"Total samples: {len(df)}")
    print(f"Unique gestures: {df['gesture'].nunique()}")
    
    # Per gesture stats
    print("\n Per Gesture Breakdown:")
    gesture_stats = df.groupby(['gesture', 'type']).size().unstack(fill_value=0)
    print(gesture_stats)
    
    # Hand distribution
    print("\n Hand Distribution:")
    hand_stats = df['hand'].value_counts()
    print(hand_stats)
    
    # File paths
    print(f"\n Dataset location: {os.path.abspath(SAVE_PATH)}")
    print(f" CSV file: {os.path.abspath(csv_path)}")
    print(f" JSON metadata: {os.path.abspath(json_path)}")
    print(f" Labels file: {os.path.abspath(labels_path)}")

# ---------------- Main ----------------
if __name__ == "__main__":
    try:
        print(" Sign Language Dataset Collector")
        print("="*50)
        
        while True:
            print("\n Menu:")
            print("1. Capture new gesture")
            print("2. Show statistics")
            print("3. List all gestures")
            print("q. Quit")
            
            choice = input("\nEnter choice: ").strip().lower()
            
            if choice == "1":
                gesture = input("Enter gesture name (English/اردو): ").strip()
                if not gesture:
                    print(" Gesture name cannot be empty")
                    continue
                    
                if gesture in metadata:
                    existing_count = len(metadata[gesture].get("files", []))
                    print(f" Gesture '{gesture}' exists with {existing_count} files.")
                    choice = input("Add more data? (y/n): ").strip().lower()
                    if choice != "y": 
                        continue

                mode = input("Press 's' for Static or 'd' for Dynamic: ").strip().lower()

                if mode == "s":
                    img_input = input(f"Number of static images (default {STATIC_IMAGES_PER_GESTURE}): ").strip()
                    runtime_static_images = int(img_input) if img_input.isdigit() else STATIC_IMAGES_PER_GESTURE
                    capture_static(gesture, runtime_static_images)

                elif mode == "d":
                    vid_input = input(f"Number of videos (default {DYNAMIC_VIDEOS_PER_GESTURE}): ").strip()
                    runtime_video_count = int(vid_input) if vid_input.isdigit() else DYNAMIC_VIDEOS_PER_GESTURE

                    dur_input = input(f"Video duration in seconds (default {VIDEO_DURATION}): ").strip()
                    runtime_video_duration = float(dur_input) if dur_input.replace('.','',1).isdigit() else VIDEO_DURATION

                    # Temporarily override for this session
                    original_count = DYNAMIC_VIDEOS_PER_GESTURE
                    DYNAMIC_VIDEOS_PER_GESTURE = runtime_video_count
                    capture_dynamic(gesture, runtime_video_duration)
                    DYNAMIC_VIDEOS_PER_GESTURE = original_count

                else:
                    print(" Invalid option, use 's' or 'd'.")
                    
            elif choice == "2":
                show_statistics()
                
            elif choice == "3":
                if metadata:
                    print("\n All Gestures:")
                    for i, (gesture, data) in enumerate(metadata.items(), 1):
                        count = len(data.get("files", []))
                        print(f"{i}. {gesture}: {count} files")
                else:
                    print("No gestures recorded yet.")
                    
            elif choice == "q":
                break
            else:
                print(" Invalid choice")
                
    except KeyboardInterrupt:
        print("\n\n Exiting...")
    finally:
        hands.close()
        cv2.destroyAllWindows()
        print(" Goodbye!")

🤟 Sign Language Dataset Collector

📋 Menu:
1. Capture new gesture
2. Show statistics
3. List all gestures
q. Quit



Enter choice:  1
Enter gesture name (English/اردو):  فوج
Press 's' for Static or 'd' for Dynamic:  s
Number of static images (default 60):  300



📸 Static capture for 'فوج' (300 images)
Controls: SPACE=pause, U=undo, Q=quit


C:\Users\AQ-PC\AppData\Roaming\Python\Python310\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


✅ Captured 1/300
✅ Captured 2/300
✅ Captured 3/300
✅ Captured 4/300
✅ Captured 5/300
✅ Captured 6/300
✅ Captured 7/300
✅ Captured 8/300
✅ Captured 9/300
✅ Captured 10/300
✅ Captured 11/300
✅ Captured 12/300
✅ Captured 13/300
✅ Captured 14/300
✅ Captured 15/300
✅ Captured 16/300
✅ Captured 17/300
✅ Captured 18/300
✅ Captured 19/300
✅ Captured 20/300
✅ Captured 21/300
✅ Captured 22/300
✅ Captured 23/300
✅ Captured 24/300
✅ Captured 25/300
✅ Captured 26/300
✅ Captured 27/300
✅ Captured 28/300
✅ Captured 29/300
✅ Captured 30/300
✅ Captured 31/300
✅ Captured 32/300
✅ Captured 33/300
✅ Captured 34/300
✅ Captured 35/300
✅ Captured 36/300
✅ Captured 37/300
✅ Captured 38/300
✅ Captured 39/300
✅ Captured 40/300
✅ Captured 41/300
✅ Captured 42/300
✅ Captured 43/300
✅ Captured 44/300
✅ Captured 45/300
✅ Captured 46/300
✅ Captured 47/300
✅ Captured 48/300
✅ Captured 49/300
✅ Captured 50/300
✅ Captured 51/300
✅ Captured 52/300
✅ Captured 53/300
✅ Captured 54/300
✅ Captured 55/300
✅ Captured 56/300
✅

...


📸 Static capture for 'میں شکست قبول کرتا ہوں' (300 images)
Controls: SPACE=pause, U=undo, Q=quit
✅ Captured 1/300
✅ Captured 2/300
✅ Captured 3/300
✅ Captured 4/300
✅ Captured 5/300
✅ Captured 6/300
✅ Captured 7/300
✅ Captured 8/300
✅ Captured 9/300
✅ Captured 10/300
✅ Captured 11/300
✅ Captured 12/300
✅ Captured 13/300
✅ Captured 14/300
✅ Captured 15/300
✅ Captured 16/300
✅ Captured 17/300
✅ Captured 18/300
✅ Captured 19/300
✅ Captured 20/300
✅ Captured 21/300
✅ Captured 22/300
✅ Captured 23/300
✅ Captured 24/300
✅ Captured 25/300
✅ Captured 26/300
✅ Captured 27/300
✅ Captured 28/300
✅ Captured 29/300
✅ Captured 30/300
✅ Captured 31/300
✅ Captured 32/300
✅ Captured 33/300
✅ Captured 34/300
✅ Captured 35/300
✅ Captured 36/300
✅ Captured 37/300
✅ Captured 38/300
✅ Captured 39/300
✅ Captured 40/300
✅ Captured 41/300
✅ Captured 42/300
✅ Captured 43/300
✅ Captured 44/300
✅ Captured 45/300
✅ Captured 46/300
✅ Captured 47/300
✅ Captured 48/300
✅ Captured 49/300
✅ Captured 50/300
✅ Captured 


Enter choice:  1
Enter gesture name (English/اردو):  خاموش
Press 's' for Static or 'd' for Dynamic:  s
Number of static images (default 60):  300



📸 Static capture for 'خاموش' (300 images)
Controls: SPACE=pause, U=undo, Q=quit
✅ Captured 1/300
✅ Captured 2/300
✅ Captured 3/300
✅ Captured 4/300
✅ Captured 5/300
✅ Captured 6/300
✅ Captured 7/300
✅ Captured 8/300
✅ Captured 9/300
✅ Captured 10/300
✅ Captured 11/300
✅ Captured 12/300
✅ Captured 13/300
✅ Captured 14/300
✅ Captured 15/300
✅ Captured 16/300
✅ Captured 17/300
✅ Captured 18/300
✅ Captured 19/300
✅ Captured 20/300
✅ Captured 21/300
✅ Captured 22/300
✅ Captured 23/300
✅ Captured 24/300
✅ Captured 25/300
✅ Captured 26/300
✅ Captured 27/300
✅ Captured 28/300
✅ Captured 29/300
✅ Captured 30/300
✅ Captured 31/300
✅ Captured 32/300
✅ Captured 33/300
✅ Captured 34/300
✅ Captured 35/300
✅ Captured 36/300
✅ Captured 37/300
✅ Captured 38/300
✅ Captured 39/300
✅ Captured 40/300
✅ Captured 41/300
✅ Captured 42/300
✅ Captured 43/300
✅ Captured 44/300
✅ Captured 45/300
✅ Captured 46/300
✅ Captured 47/300
✅ Captured 48/300
✅ Captured 49/300
✅ Captured 50/300
✅ Captured 51/300
✅ Captured


Enter choice:  1
Enter gesture name (English/اردو):  آپ کہاں رہتے ہیں؟
Press 's' for Static or 'd' for Dynamic:  d
Number of videos (default 10):  27
Video duration in seconds (default 7):  6



🎬 Dynamic capture for 'آپ کہاں رہتے ہیں؟' (27 videos of 6.0s each)

📹 Video 1/27
⏳ Get ready... recording starts in 2s
✋ Recording started!
✅ Video 1/27 saved!

⏸ Resting 5s before next video...

📹 Video 2/27
⏳ Get ready... recording starts in 2s
✋ Recording started!
✅ Video 2/27 saved!

⏸ Resting 5s before next video...

📹 Video 3/27
⏳ Get ready... recording starts in 2s
✋ Recording started!
✅ Video 3/27 saved!

⏸ Resting 5s before next video...

📹 Video 4/27
⏳ Get ready... recording starts in 2s
✋ Recording started!
✅ Video 4/27 saved!

⏸ Resting 5s before next video...

📹 Video 5/27
⏳ Get ready... recording starts in 2s
✋ Recording started!
